In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
import torch
import math

# **self-attention class**

In [4]:
class SelfAttention:

    # to initialize it
    def __init__(self, dimen=4, heads=1, random_w=True):

        self.dimen = dimen
        self.random_w = random_w
        self.heads = heads

        if random_w:
            self.w_query = torch.rand(dimen, dimen//heads).float()
            self.w_key = torch.rand(dimen, dimen//heads).float()
            self.w_value = torch.rand(dimen, dimen//heads).float()
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, static_embeddings):

        if self.random_w == False:
            
            queries = static_embeddings.float()
            keys = static_embeddings.float()
            values = static_embeddings.float()
            
        else:
        
            queries = torch.matmul(static_embeddings.float(), self.w_query)
            keys = torch.matmul(static_embeddings.float(), self.w_key)
            values = torch.matmul(static_embeddings.float(), self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)

        #print(contextual_embeddings)
        return contextual_embeddings
        

    # get embeddings of different sentences
    def __call__(self, static_embeddings):

        # 1. obtain query, key, value vectors
        queries, keys, values = self.getQKV(static_embeddings)

        # 2. perform dot product bw query and key
        dot_product = self.dotproduct(queries,keys)

        # 3. scale by 1/sqrt(dimen)
        scaled_dp = self.get_sqrt(dot_product)

        # 4. apply softmax to each row
        weights = self.apply_softmax(scaled_dp)

        # 5. multiply w value to get final embeddings
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        return contextual_embeddings

In [5]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [6]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [7]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [8]:
model = Word2Vec(sentences, vector_size=256, window=5, min_count=1, sg=1)

In [9]:
input_matrix1 = []

for word in s1:
    embedding = model.wv[word]
    input_matrix1.append(embedding)

np_ip1 = np.array(input_matrix1)
static_embeddings1 = torch.tensor(np_ip1)

#print(query_tensors)

In [10]:
sa_block = SelfAttention(256,True)

In [11]:
ce1 = sa_block(static_embeddings1)

In [12]:
input_matrix2 = []

for word in s2:
    embedding = model.wv[word]
    input_matrix2.append(embedding)

np_ip2 = np.array(input_matrix2)
static_embeddings2 = torch.tensor(np_ip2)

In [13]:
ce2 = sa_block(static_embeddings2)

In [14]:
print(static_embeddings2[0:2])

tensor([[-2.8321e-03, -3.7513e-03, -1.0717e-03, -3.2667e-03, -2.3589e-03,
         -2.2152e-03, -9.1568e-04, -6.6680e-04, -3.4988e-03, -2.8719e-04,
          3.1846e-03,  3.0041e-03, -2.8149e-03, -1.4324e-03,  1.2182e-03,
         -3.7386e-03,  5.7673e-04,  2.5486e-03,  2.2447e-03, -3.4231e-03,
         -1.7645e-03, -3.1798e-03,  1.7952e-05,  3.6186e-03,  2.3333e-03,
          1.9794e-03,  1.9770e-03, -1.2668e-03,  3.7313e-03, -2.8736e-03,
         -2.8400e-03, -8.8492e-04, -3.0413e-04, -1.2563e-03, -2.3148e-04,
          2.9253e-03, -2.7247e-04, -6.3474e-04,  1.0720e-03, -3.2653e-03,
          3.0687e-03,  3.3344e-03, -3.7438e-03,  9.5557e-04,  3.8691e-03,
         -2.9945e-03, -2.7215e-03, -3.0221e-03,  3.2797e-03, -2.6615e-04,
          3.5720e-03, -3.1868e-03,  1.4621e-03,  1.0293e-03,  2.9012e-04,
          9.0925e-04, -2.9176e-03, -3.6556e-03,  9.1976e-04,  2.4017e-03,
          3.1194e-03,  2.2406e-03, -3.0365e-04,  3.2446e-03, -3.6470e-03,
          1.3305e-03,  1.0420e-04,  1.

# **multi-head attention**

In [15]:
ambiguous_sentence = "The cat chased the mouse until it stumbled"
ambiguous_sentence = ambiguous_sentence.split(" ")
print(ambiguous_sentence)

['The', 'cat', 'chased', 'the', 'mouse', 'until', 'it', 'stumbled']


In [16]:
def get_embeddings(model, sentence):
    
    embeddings = [model.wv[word] for word in sentence]
    embeddings_np = np.array(embeddings)
    return torch.tensor(embeddings_np)

In [17]:
model = Word2Vec([ambiguous_sentence], vector_size=512, window=5, min_count=1, sg=1)

In [18]:
emb = get_embeddings(model,ambiguous_sentence)
print(emb.shape)

torch.Size([8, 512])


In [19]:
print(emb)

tensor([[-8.3858e-04, -1.2931e-03, -2.7027e-04,  ..., -1.9349e-03,
         -2.9324e-04,  1.3476e-03],
        [ 1.2903e-03, -2.2377e-04,  1.4973e-03,  ...,  1.0317e-04,
         -1.7566e-03,  1.6347e-03],
        [ 1.1490e-03, -5.7839e-04,  6.1753e-04,  ..., -1.6250e-03,
         -2.8566e-05, -5.1714e-04],
        ...,
        [-1.2890e-03,  8.4774e-04, -9.2673e-05,  ...,  1.8470e-03,
         -1.1359e-03,  1.6143e-03],
        [-1.4161e-03, -1.8756e-03, -5.3587e-04,  ..., -5.0609e-04,
          1.4158e-03, -6.7645e-04],
        [-1.0473e-04,  4.6178e-05,  9.9675e-04,  ..., -1.3424e-03,
         -9.7645e-04, -4.4665e-04]])


In [20]:
sa1 = SelfAttention(512,8,True)
sa2 = SelfAttention(512,8,True)
sa3 = SelfAttention(512,8,True)
sa4 = SelfAttention(512,8,True)
sa5 = SelfAttention(512,8,True)
sa6 = SelfAttention(512,8,True)
sa7 = SelfAttention(512,8,True)
sa8 = SelfAttention(512,8,True)

In [21]:
e1 = sa1(emb)
e2 = sa2(emb)
e3 = sa3(emb)
e4 = sa4(emb)
e5 = sa5(emb)
e6 = sa6(emb)
e7 = sa7(emb)
e8 = sa8(emb)

In [22]:
print(e1.shape)

torch.Size([8, 64])


In [23]:
print(e2)

tensor([[ 0.0045,  0.0022,  0.0045,  0.0032,  0.0069,  0.0062,  0.0045,  0.0050,
          0.0078,  0.0046,  0.0023,  0.0016,  0.0039,  0.0062,  0.0033,  0.0025,
          0.0061, -0.0015,  0.0048,  0.0059,  0.0025,  0.0027,  0.0067,  0.0048,
          0.0033,  0.0019,  0.0006,  0.0050,  0.0034,  0.0057,  0.0060,  0.0055,
          0.0043,  0.0047,  0.0008,  0.0027, -0.0012,  0.0092,  0.0028,  0.0049,
         -0.0025,  0.0061,  0.0067,  0.0064,  0.0047,  0.0023,  0.0034,  0.0015,
          0.0093,  0.0063,  0.0068,  0.0046,  0.0018,  0.0030,  0.0066,  0.0060,
          0.0024,  0.0071,  0.0082,  0.0067,  0.0045,  0.0077,  0.0089,  0.0072],
        [ 0.0045,  0.0022,  0.0045,  0.0032,  0.0069,  0.0062,  0.0045,  0.0050,
          0.0078,  0.0046,  0.0023,  0.0016,  0.0039,  0.0062,  0.0033,  0.0025,
          0.0061, -0.0015,  0.0048,  0.0059,  0.0025,  0.0027,  0.0067,  0.0048,
          0.0033,  0.0019,  0.0006,  0.0050,  0.0034,  0.0057,  0.0060,  0.0055,
          0.0043,  0.0047, 

In [24]:
concatenated = torch.cat([e1,e2,e3,e4,e5,e6,e7,e8], dim=1)
print(concatenated.shape)

torch.Size([8, 512])


In [25]:
print(concatenated)

tensor([[0.0036, 0.0042, 0.0080,  ..., 0.0046, 0.0038, 0.0028],
        [0.0036, 0.0042, 0.0080,  ..., 0.0046, 0.0038, 0.0028],
        [0.0036, 0.0042, 0.0080,  ..., 0.0045, 0.0038, 0.0028],
        ...,
        [0.0036, 0.0042, 0.0080,  ..., 0.0046, 0.0038, 0.0028],
        [0.0036, 0.0042, 0.0080,  ..., 0.0046, 0.0038, 0.0028],
        [0.0036, 0.0042, 0.0080,  ..., 0.0046, 0.0038, 0.0028]])


In [26]:
class MultiHeadAttention:
    
    def __init__(self, num_heads=1, dim=4, random_w=True):
        
        self.num_heads = num_heads
        self.dim = dim
        self.random_w = random_w

    def __call__(self, emb):
        
        outputs = []
        
        for i in range(self.num_heads):
            sa = SelfAttention(self.dim, self.num_heads, self.random_w)
            ce = sa(emb)
            outputs.append(ce)

        concatenated = torch.cat(outputs, dim=1)
        return concatenated

In [27]:
ma = MultiHeadAttention(8,512,True)
embb = ma(emb)
print(embb.shape)

torch.Size([8, 512])


In [28]:
print(embb)

tensor([[0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0037, 0.0018],
        [0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0038, 0.0018],
        [0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0037, 0.0018],
        ...,
        [0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0038, 0.0018],
        [0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0037, 0.0018],
        [0.0073, 0.0034, 0.0031,  ..., 0.0092, 0.0038, 0.0018]])


# **positional encoding**

In [29]:
print(math.sin(1))
print(math.cos(1))

0.8414709848078965
0.5403023058681398


In [30]:
for i in range(2):
    print(i)

0
1


In [31]:
pos = 1
dim = 6

for word in emb:
    pos_enc = []
    for i in range(dim//2): # 1 pair
        pos_enc.append(math.sin(pos/pow(10000,(2*i/dim))))
        pos_enc.append(math.cos(pos/pow(10000,(2*i/dim))))
    print(pos_enc)

[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792064809]
[0.8414709848078965, 0.5403023058681398, 0.046399223464731285, 0.9989229760406304, 0.0021544330233656045, 0.9999976792

In [32]:
def get_emb(model, word, pos, dim):
        
    emb = model.wv[word]
    
    pos_enc = np.zeros(dim)
    
    for i in range(dim//2): # 1 pair
        pos_enc[2*i] = math.sin(pos/pow(10000,(2*i/dim)))
        pos_enc[2*i+1] = math.cos(pos/pow(10000,(2*i/dim)))

    emb_np = np.array(emb)
    emb_t = torch.tensor(emb_np)

    pos_np = np.array(pos_enc)
    pos_t = torch.tensor(pos_np)

    result = torch.add(emb_t, pos_t)
    return result


def generate_pos_encodings(model,sentence):

    embeddings = [get_emb(model,word,pos,512) for pos,word in enumerate(sentence)]
    return torch.stack(embeddings)

In [33]:
enc = generate_pos_encodings(model,ambiguous_sentence)
print(enc.shape)

torch.Size([8, 512])


In [34]:
apply_ma = MultiHeadAttention(8,512,True)
get_emb = apply_ma(enc)
print(get_emb.shape)

torch.Size([8, 512])
